# MIMIC-IV 30-Day Readmission — Data Representation

Development notebook for the production MLOps pipeline on GCP.

**Scope of this notebook:** build the BigQuery data representation only —
cohort, label, feature views, and clinical baselines. Modeling and Vertex AI
orchestration live in separate artifacts.

**Working principle:** one cell at a time. Each cell has a single, named
purpose. Open design questions are tracked in
[`docs/open_questions.md`](docs/open_questions.md).

## Cell 1 — Setup & configuration

Single source of truth for project, datasets, and the BigQuery client.
No queries are executed here.

In [ ]:
from google.cloud import bigquery

# --- Project & location -------------------------------------------------
PROJECT_ID = "enterprise-clinical-copilot"
LOCATION = "US"

# --- MIMIC-IV source (PhysioNet public BigQuery) ------------------------
# Datasets confirmed available in this GCP project:
#   physionet-data.mimiciv_3_1_hosp     - core hospital tables (versioned)
#   physionet-data.mimiciv_3_1_icu      - ICU tables           (versioned)
#   physionet-data.mimiciv_3_1_derived  - mimic-code concepts  (versioned)
#   physionet-data.mimiciv_ed           - ED module            (unversioned)
#   physionet-data.mimiciv_note         - clinical notes       (unversioned)
MIMIC_VERSION = "v3_1"

SOURCE_PROJECT = "physionet-data"
SOURCE_HOSP = f"{SOURCE_PROJECT}.mimiciv_3_1_hosp"
SOURCE_ICU = f"{SOURCE_PROJECT}.mimiciv_3_1_icu"
SOURCE_DERIVED = f"{SOURCE_PROJECT}.mimiciv_3_1_derived"
SOURCE_ED = f"{SOURCE_PROJECT}.mimiciv_ed"
SOURCE_NOTE = f"{SOURCE_PROJECT}.mimiciv_note"

# --- Curated destination ------------------------------------------------
DEST_DATASET = "readmission"
DEST = f"{PROJECT_ID}.{DEST_DATASET}"

# --- BigQuery client ----------------------------------------------------
bq = bigquery.Client(project=PROJECT_ID, location=LOCATION)

print(f"Project:       {PROJECT_ID}")
print(f"Location:      {LOCATION}")
print(f"MIMIC hosp:    {SOURCE_HOSP}")
print(f"MIMIC icu:     {SOURCE_ICU}")
print(f"MIMIC derived: {SOURCE_DERIVED}")
print(f"MIMIC ed:      {SOURCE_ED}")
print(f"MIMIC note:    {SOURCE_NOTE}")
print(f"Curated dest:  {DEST}")


## Part 1 — Eligible-admission cohort

Builds `readmission.cohort_admissions` (**one row per eligible
admission**, not per patient).

Rationale for keeping every eligible admission rather than only the
earliest per patient: the "first stay only" rule mathematically guarantees
that every prior-utilization feature (LACE-E, HOSPITAL-A, rolling
admission/ED counts) collapses to zero, since by construction no patient
has any prior history. Empirically this dragged the LACE/HOSPITAL
discriminative floor below the published 0.60–0.68 band. Standard
MIMIC-IV readmission benchmarks (e.g. Adisa et al.) retain ~415K
admissions and find prior-admission count is the single strongest
predictor — the very signal the first-stay rule deletes.

Eligibility filters, applied in order:

1. **Adult**: `anchor_age + (YEAR(admittime) - anchor_year) >= 18`.
2. **LOS ≥ 24h**: `TIMESTAMP_DIFF(dischtime, admittime, HOUR) >= 24`.
3. **Mortality exclusion**: `hospital_expire_flag = 0 AND deathtime IS NULL`.
4. **Discharge disposition exclusion**: `discharge_location` not in
   `HOSPICE`, `AGAINST ADVICE`, `OTHER FACILITY`, `ACUTE HOSPITAL`.

`admission_type` is **not** filtered here — it only affects the label
(qualifying readmission triggers in Part 2).

**Leakage discipline:** because the same `subject_id` now appears multiple
times, all train / validation / test splits downstream **must** be grouped
by `subject_id` (`GroupKFold` / `GroupShuffleSplit` on patient ID). A
random `hadm_id`-level split would place the same patient in both train
and test and produce inflated metrics.


In [ ]:
# Cell 2 — Ensure destination dataset exists (idempotent, one-time).
from google.cloud.exceptions import NotFound

_dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DEST_DATASET}")
_dataset_ref.location = LOCATION
_dataset_ref.description = "Curated tables for MIMIC-IV 30-day readmission pipeline."

try:
    bq.get_dataset(_dataset_ref)
    print(f"Dataset already exists: {DEST}")
except NotFound:
    bq.create_dataset(_dataset_ref)
    print(f"Created dataset: {DEST}")


In [ ]:
# Cell 3 — Build the eligible-admission cohort (Part 1, steps 1-4).
#
# One row per eligible hadm_id (NOT per subject_id).  All four filters are
# applied; we deliberately do NOT collapse to a single admission per patient,
# because doing so erases historical-utilization signal (LACE-E, HOSPITAL-A,
# rolling counts) and depresses every downstream baseline.  Per-patient
# leakage is handled later by GroupKFold/GroupShuffleSplit on subject_id.
#
# Physical layout: MIMIC-IV randomly shifts each patient's dates across a
# ~100-year span, so DAY-level partitioning on admittime exceeds BigQuery's
# 4,000-partition limit.  We cluster by subject_id, the join key for every
# downstream feature view and the grouping key for splits.
COHORT_TABLE = f"{DEST}.cohort_admissions"

cohort_sql = f"""
CREATE OR REPLACE TABLE `{COHORT_TABLE}`
CLUSTER BY subject_id AS
WITH base AS (
  SELECT
    a.subject_id,
    a.hadm_id,
    a.admittime,
    a.dischtime,
    a.admission_type,
    a.admission_location,
    a.discharge_location,
    a.insurance,
    a.language,
    a.marital_status,
    a.race,
    a.edregtime,
    a.edouttime,
    p.gender,
    p.anchor_age,
    p.anchor_year,
    p.anchor_year_group,
    p.anchor_age + (EXTRACT(YEAR FROM a.admittime) - p.anchor_year) AS age_at_admission,
    TIMESTAMP_DIFF(a.dischtime, a.admittime, HOUR) AS los_hours
  FROM `{SOURCE_HOSP}.admissions` a
  JOIN `{SOURCE_HOSP}.patients`   p USING (subject_id)
  WHERE a.admittime IS NOT NULL
    AND a.dischtime IS NOT NULL
    -- Step 3: mortality exclusion
    AND a.hospital_expire_flag = 0
    AND a.deathtime IS NULL
    -- Step 4: discharge disposition exclusion (NULLs retained)
    AND (
      a.discharge_location IS NULL
      OR a.discharge_location NOT IN (
        'HOSPICE', 'AGAINST ADVICE', 'OTHER FACILITY', 'ACUTE HOSPITAL'
      )
    )
)
SELECT *
FROM base
WHERE age_at_admission >= 18    -- Step 1
  AND los_hours          >= 24  -- Step 2
"""

job = bq.query(cohort_sql)
job.result()  # block until complete
table = bq.get_table(COHORT_TABLE)

# --- Diagnostics: rows, distinct patients, mean stays per patient -------
diag_sql = f"""
WITH per_patient AS (
  SELECT subject_id, COUNT(*) AS n
  FROM `{COHORT_TABLE}`
  GROUP BY subject_id
)
SELECT
  (SELECT COUNT(*)                FROM `{COHORT_TABLE}`) AS n_admissions,
  COUNT(*)                                               AS n_patients,
  ROUND(AVG(n), 2)                                       AS mean_admits_per_patient,
  MAX(n)                                                 AS max_admits_one_patient,
  COUNTIF(n > 1)                                         AS n_patients_multi_admit
FROM per_patient
"""
d = bq.query(diag_sql).result().to_dataframe().iloc[0]

print(f"Built {COHORT_TABLE}")
print(f"  rows (admissions):        {table.num_rows:,}")
print(f"  bytes:                    {table.num_bytes / 1e6:,.1f} MB")
print(f"  distinct patients:        {int(d.n_patients):,}")
print(f"  mean admits / patient:    {d.mean_admits_per_patient}")
print(f"  max admits / patient:     {int(d.max_admits_one_patient)}")
print(f"  patients with >=2 admits: {int(d.n_patients_multi_admit):,}")
print(f"  cluster col:              subject_id")


## Part 2 — Target label (30-day unplanned readmission)

Builds `readmission.cohort_labeled` (one row per eligible admission =
cohort + label).

For each index admission, scan the raw `admissions` table for the same
`subject_id` and apply, in order:


In [ ]:
# Cell 4 — Build the labeled cohort (Part 2, steps 1-4).
LABELED_TABLE = f"{DEST}.cohort_labeled"

ACUTE_TYPES = (
    "'URGENT'",
    "'EMERGENCY'",
    "'EW EMER.'",
    "'DIRECT EMER.'",
    "'DIRECT OBSERVATION'",
    "'EU OBSERVATION'",
    "'OBSERVATION ADMIT'",
    "'AMBULATORY OBSERVATION'",
)
acute_types_sql = ", ".join(ACUTE_TYPES)

label_sql = f"""
CREATE OR REPLACE TABLE `{LABELED_TABLE}`
CLUSTER BY subject_id AS
WITH subsequent AS (
  -- Step 1+2: for each index, find later admissions within 30 days.
  -- Step 3: acute flag on each candidate.
  SELECT
    c.subject_id,
    c.hadm_id                                                      AS index_hadm_id,
    DATE_DIFF(DATE(n.admittime), DATE(c.dischtime), DAY)           AS days_to_readmit,
    n.admission_type                                               AS next_admission_type,
    n.admission_type IN ({acute_types_sql})                        AS is_acute
  FROM `{COHORT_TABLE}`              c
  JOIN `{SOURCE_HOSP}.admissions`    n
    ON n.subject_id = c.subject_id
   AND n.hadm_id   != c.hadm_id
   AND n.admittime  > c.dischtime
   AND DATE_DIFF(DATE(n.admittime), DATE(c.dischtime), DAY) BETWEEN 1 AND 30
),
agg AS (
  -- Step 4: collapse to one row per index admission.
  -- Pick the EARLIEST qualifying acute readmission for the diagnostic columns.
  SELECT
    subject_id,
    index_hadm_id,
    COUNTIF(TRUE)                                                  AS n_readmits_30d_any,
    LOGICAL_OR(is_acute)                                           AS has_acute_30d,
    MIN(IF(is_acute, days_to_readmit, NULL))                       AS days_to_readmit,
    ARRAY_AGG(
      IF(is_acute, next_admission_type, NULL) IGNORE NULLS
      ORDER BY days_to_readmit
      LIMIT 1
    )[SAFE_OFFSET(0)]                                              AS next_admission_type
  FROM subsequent
  GROUP BY subject_id, index_hadm_id
)
SELECT
  c.*,
  CAST(COALESCE(a.has_acute_30d, FALSE) AS INT64) AS label,
  COALESCE(a.n_readmits_30d_any, 0)               AS n_readmits_30d_any,
  a.days_to_readmit,
  a.next_admission_type
FROM `{COHORT_TABLE}` c
LEFT JOIN agg a
  ON a.subject_id    = c.subject_id
 AND a.index_hadm_id = c.hadm_id
"""

bq.query(label_sql).result()

# --- Sanity-check the label distribution --------------------------------
stats_sql = f"""
SELECT
  COUNT(*)                                                        AS n_total,
  COUNTIF(label = 1)                                              AS n_positive,
  ROUND(SAFE_DIVIDE(COUNTIF(label = 1), COUNT(*)) * 100, 2)       AS positive_rate_pct,
  COUNTIF(n_readmits_30d_any > 0 AND label = 0)                   AS n_30d_elective_only,
  AVG(IF(label = 1, days_to_readmit, NULL))                       AS mean_days_to_readmit
FROM `{LABELED_TABLE}`
"""
stats = bq.query(stats_sql).result().to_dataframe().iloc[0]

table = bq.get_table(LABELED_TABLE)
print(f"Built {LABELED_TABLE}")
print(f"  rows:                    {table.num_rows:,}")
print(f"  positives (y=1):         {int(stats.n_positive):,}")
print(f"  prevalence:              {stats.positive_rate_pct:.2f}%")
print(f"  in-window elective-only: {int(stats.n_30d_elective_only):,}  (kept as y=0)")
print(f"  mean days to readmit:    {stats.mean_days_to_readmit:.1f}")


## Part 3 — Clinical baselines (LACE & HOSPITAL)

These two scores establish the **discriminative floor** any ML model must
beat. They are not features; they are competing models built from
deterministic rules in the literature.

Both are computed strictly from information knowable **at index `dischtime`**
and are anchored to the row keys (`subject_id`, `hadm_id`) of
`cohort_labeled` so we can join them back later.

### Cell 5 — LACE index (0–19 points)

| Letter | Component                | Source                                          |
|--------|--------------------------|-------------------------------------------------|
| **L**  | Length of stay (days)    | `cohort_labeled.los_hours`                      |
| **A**  | Acuity (acute admit?)    | `cohort_labeled.admission_type`                 |
| **C**  | Charlson comorbidity idx | `mimiciv_3_1_derived.charlson` (pre-computed)   |
| **E**  | ED visits, prior 180 d   | `mimiciv_3_1_hosp.admissions.edregtime`         |

Acute admit set is the same eight `admission_type` values used for the
30-day label. Prior ED count excludes the index admission itself.


In [ ]:
# Cell 5 — Build the LACE index per index admission.
LACE_TABLE = f"{DEST}.cohort_lace"

# Reuse the same acute-trigger set we pinned for the 30-day label so LACE-A
# and the label share a single definition of "unplanned."
acute_types_sql = ", ".join(ACUTE_TYPES)

lace_sql = f"""
CREATE OR REPLACE TABLE `{LACE_TABLE}`
CLUSTER BY subject_id AS
WITH
-- E: count of prior ED-routed admissions in the 180 days BEFORE index admittime.
-- Excludes the index hadm_id itself; uses edregtime IS NOT NULL as the
-- "ED visit" marker per the instructions.
prior_ed AS (
  SELECT
    c.subject_id,
    c.hadm_id,
    COUNTIF(
      h.edregtime IS NOT NULL
      AND h.hadm_id != c.hadm_id
      AND h.admittime <  c.admittime
      AND h.admittime >= TIMESTAMP_SUB(c.admittime, INTERVAL 180 DAY)
    ) AS n_ed_180d
  FROM `{COHORT_TABLE}`              c
  LEFT JOIN `{SOURCE_HOSP}.admissions` h
    ON h.subject_id = c.subject_id
  GROUP BY c.subject_id, c.hadm_id
),
-- C: pre-computed Charlson comorbidity index from mimiciv_derived.
-- LEFT JOIN so admissions without a derived row score 0 (treated as no
-- recorded comorbidities, matching the score's "missing = normal" stance).
charlson AS (
  SELECT hadm_id, charlson_comorbidity_index AS cci
  FROM `{SOURCE_DERIVED}.charlson`
),
scored AS (
  SELECT
    c.subject_id,
    c.hadm_id,
    -- raw inputs (kept for auditability)
    CAST(FLOOR(c.los_hours / 24) AS INT64) AS los_days,
    c.admission_type,
    COALESCE(ch.cci, 0)                    AS cci,
    COALESCE(pe.n_ed_180d, 0)              AS n_ed_180d,

    -- L: length of stay (days)
    CASE
      WHEN c.los_hours / 24 <  1  THEN 0
      WHEN c.los_hours / 24 <  2  THEN 1   -- 1 day
      WHEN c.los_hours / 24 <  3  THEN 2   -- 2 days
      WHEN c.los_hours / 24 <  4  THEN 3   -- 3 days
      WHEN c.los_hours / 24 <  7  THEN 4   -- 4-6 days
      WHEN c.los_hours / 24 < 14  THEN 5   -- 7-13 days
      ELSE                              7  -- >=14 days
    END AS lace_l,

    -- A: acuity (3 if acute/unplanned, else 0)
    IF(c.admission_type IN ({acute_types_sql}), 3, 0) AS lace_a,

    -- C: Charlson buckets
    CASE
      WHEN COALESCE(ch.cci, 0) = 0 THEN 0
      WHEN ch.cci = 1              THEN 1
      WHEN ch.cci = 2              THEN 2
      WHEN ch.cci = 3              THEN 3
      ELSE                              5  -- >=4
    END AS lace_c,

    -- E: prior ED visits in 180d
    CASE
      WHEN COALESCE(pe.n_ed_180d, 0) = 0 THEN 0
      WHEN pe.n_ed_180d = 1              THEN 1
      WHEN pe.n_ed_180d = 2              THEN 2
      WHEN pe.n_ed_180d = 3              THEN 3
      ELSE                                    4  -- >=4
    END AS lace_e
  FROM `{COHORT_TABLE}` c
  LEFT JOIN charlson  ch USING (hadm_id)
  LEFT JOIN prior_ed  pe USING (subject_id, hadm_id)
)
SELECT
  *,
  lace_l + lace_a + lace_c + lace_e AS lace_score
FROM scored
"""

bq.query(lace_sql).result()

# --- Sanity check: distribution + component means ----------------------
stats_sql = f"""
SELECT
  COUNT(*)                  AS n_rows,
  MIN(lace_score)           AS min_score,
  MAX(lace_score)           AS max_score,
  ROUND(AVG(lace_score), 2) AS mean_score,
  ROUND(AVG(lace_l), 2)     AS mean_l,
  ROUND(AVG(lace_a), 2)     AS mean_a,
  ROUND(AVG(lace_c), 2)     AS mean_c,
  ROUND(AVG(lace_e), 2)     AS mean_e,
  COUNTIF(cci IS NULL OR cci = 0) AS n_cci_zero_or_missing
FROM `{LACE_TABLE}`
"""
s = bq.query(stats_sql).result().to_dataframe().iloc[0]

print(f"Built {LACE_TABLE}")
print(f"  rows:           {int(s.n_rows):,}")
print(f"  score range:    {int(s.min_score)}–{int(s.max_score)}  (theoretical 0–19)")
print(f"  mean score:     {s.mean_score}")
print(f"  component means L/A/C/E: {s.mean_l} / {s.mean_a} / {s.mean_c} / {s.mean_e}")
print(f"  CCI = 0 or missing: {int(s.n_cci_zero_or_missing):,}")


### Cell 6 — HOSPITAL score (0–13) + combined baselines + evaluation

| Letter | Component                              | Source                                                |
|--------|----------------------------------------|-------------------------------------------------------|
| **H**  | last Hgb < 12 g/dL                     | `mimiciv_3_1_derived.complete_blood_count`            |
| **O**  | oncology (service OR active cancer)    | `mimiciv_3_1_hosp.services` ∪ `diagnoses_icd`         |
| **S**  | last Na < 135 mEq/L                    | `mimiciv_3_1_derived.chemistry`                       |
| **P**  | any ICD procedure during stay          | `mimiciv_3_1_hosp.procedures_icd`                     |
| **I**  | non-elective admission                 | `cohort_labeled.admission_type`                       |
| **A**  | distinct admissions in prior 365 d     | `mimiciv_3_1_hosp.admissions.admittime`               |
| **L**  | LOS ≥ 5 days                           | `cohort_labeled.los_hours`                            |

**Conventions:** missing labs → 0 points (HOSPITAL paper: "missing = normal").
H/S use the last value with `charttime < dischtime` (strict, no leakage).
Oncology fires if **either** the discharging service is `OMED` **or** any
diagnosis on the index stay matches active-malignancy ICD codes
(ICD-9 `140`–`209`, ICD-10 `C*` or `D00`–`D09`; personal-history codes
`V10`/`Z85` and benign neoplasms excluded).

After building the HOSPITAL table, this cell joins LACE + HOSPITAL +
labels into `readmission.cohort_baselines` and computes AUROC / AUPRC
for both scores against `label`.


In [ ]:
# Cell 6 — HOSPITAL score, combined baselines table, sklearn evaluation.
from sklearn.metrics import roc_auc_score, average_precision_score

HOSPITAL_TABLE  = f"{DEST}.cohort_hospital"
BASELINES_TABLE = f"{DEST}.cohort_baselines"

hospital_sql = f"""
CREATE OR REPLACE TABLE `{HOSPITAL_TABLE}`
CLUSTER BY subject_id AS
WITH
-- H: last hemoglobin strictly before dischtime.
last_hgb AS (
  SELECT
    c.subject_id,
    c.hadm_id,
    ARRAY_AGG(cbc.hemoglobin IGNORE NULLS
              ORDER BY cbc.charttime DESC LIMIT 1)[SAFE_OFFSET(0)] AS hgb
  FROM `{COHORT_TABLE}` c
  LEFT JOIN `{SOURCE_DERIVED}.complete_blood_count` cbc
    ON cbc.hadm_id   = c.hadm_id
   AND cbc.charttime < c.dischtime
  GROUP BY c.subject_id, c.hadm_id
),
-- S: last sodium strictly before dischtime.
last_na AS (
  SELECT
    c.subject_id,
    c.hadm_id,
    ARRAY_AGG(chem.sodium IGNORE NULLS
              ORDER BY chem.charttime DESC LIMIT 1)[SAFE_OFFSET(0)] AS na
  FROM `{COHORT_TABLE}` c
  LEFT JOIN `{SOURCE_DERIVED}.chemistry` chem
    ON chem.hadm_id   = c.hadm_id
   AND chem.charttime < c.dischtime
  GROUP BY c.subject_id, c.hadm_id
),
-- O part 1: discharging service = OMED.
-- "Discharging" = the latest services row whose transfertime <= dischtime.
disch_service AS (
  SELECT subject_id, hadm_id, curr_service
  FROM (
    SELECT
      c.subject_id,
      c.hadm_id,
      s.curr_service,
      ROW_NUMBER() OVER (
        PARTITION BY c.hadm_id
        ORDER BY s.transfertime DESC
      ) AS rn
    FROM `{COHORT_TABLE}` c
    JOIN `{SOURCE_HOSP}.services` s
      ON s.hadm_id      = c.hadm_id
     AND s.transfertime <= c.dischtime
  )
  WHERE rn = 1
),
-- O part 2: any active-malignancy ICD on the index stay.
-- ICD-9 chapter "Neoplasms": 140-209 (excludes V10 personal history & 210-229 benign).
-- ICD-10: C* (all malignant) and D00-D09 (carcinoma in situ); excludes Z85.
cancer_dx AS (
  SELECT DISTINCT c.hadm_id
  FROM `{COHORT_TABLE}` c
  JOIN `{SOURCE_HOSP}.diagnoses_icd` d
    ON d.hadm_id = c.hadm_id
  WHERE
    (d.icd_version = 9  AND SUBSTR(d.icd_code, 1, 3) BETWEEN '140' AND '209')
    OR
    (d.icd_version = 10 AND (
        STARTS_WITH(d.icd_code, 'C')
        OR SUBSTR(d.icd_code, 1, 3) BETWEEN 'D00' AND 'D09'
    ))
),
-- P: any procedures_icd row attached to the index hadm_id.
any_proc AS (
  SELECT DISTINCT c.hadm_id
  FROM `{COHORT_TABLE}` c
  JOIN `{SOURCE_HOSP}.procedures_icd` p
    ON p.hadm_id = c.hadm_id
),
-- A: distinct prior admissions in the 365 days BEFORE index admittime.
-- Excludes the index hadm_id; counts each hadm_id once.
prior_adm AS (
  SELECT
    c.subject_id,
    c.hadm_id,
    COUNT(DISTINCT h.hadm_id) AS n_adm_365d
  FROM `{COHORT_TABLE}` c
  LEFT JOIN `{SOURCE_HOSP}.admissions` h
    ON h.subject_id = c.subject_id
   AND h.hadm_id   != c.hadm_id
   AND h.admittime <  c.admittime
   AND h.admittime >= TIMESTAMP_SUB(c.admittime, INTERVAL 365 DAY)
  GROUP BY c.subject_id, c.hadm_id
),
scored AS (
  SELECT
    c.subject_id,
    c.hadm_id,
    -- raw inputs for auditability
    lh.hgb,
    ln.na,
    ds.curr_service                                    AS disch_service,
    cd.hadm_id IS NOT NULL                             AS has_cancer_dx,
    ap.hadm_id IS NOT NULL                             AS has_any_procedure,
    COALESCE(pa.n_adm_365d, 0)                         AS n_adm_365d,
    c.admission_type,
    c.los_hours,

    -- H: 1 if Hgb < 12 (missing => 0)
    IF(lh.hgb IS NOT NULL AND lh.hgb < 12, 1, 0)       AS hosp_h,

    -- O: 2 if discharging service OMED OR active cancer ICD
    IF(ds.curr_service = 'OMED' OR cd.hadm_id IS NOT NULL, 2, 0) AS hosp_o,

    -- S: 1 if Na < 135 (missing => 0)
    IF(ln.na IS NOT NULL AND ln.na < 135, 1, 0)        AS hosp_s,

    -- P: 1 if any ICD procedure on index stay
    IF(ap.hadm_id IS NOT NULL, 1, 0)                   AS hosp_p,

    -- I: 1 if non-elective admission (matches LACE-A acute set)
    IF(c.admission_type IN ({acute_types_sql}), 1, 0)  AS hosp_i,

    -- A: prior admissions buckets
    CASE
      WHEN COALESCE(pa.n_adm_365d, 0) <= 1 THEN 0
      WHEN pa.n_adm_365d <= 5              THEN 2
      ELSE                                       5
    END                                                AS hosp_a,

    -- L: 2 if LOS >= 5 days
    IF(c.los_hours / 24 >= 5, 2, 0)                    AS hosp_l
  FROM `{COHORT_TABLE}`     c
  LEFT JOIN last_hgb        lh ON lh.subject_id = c.subject_id AND lh.hadm_id = c.hadm_id
  LEFT JOIN last_na         ln ON ln.subject_id = c.subject_id AND ln.hadm_id = c.hadm_id
  LEFT JOIN disch_service   ds ON ds.subject_id = c.subject_id AND ds.hadm_id = c.hadm_id
  LEFT JOIN cancer_dx       cd ON cd.hadm_id = c.hadm_id
  LEFT JOIN any_proc        ap ON ap.hadm_id = c.hadm_id
  LEFT JOIN prior_adm       pa ON pa.subject_id = c.subject_id AND pa.hadm_id = c.hadm_id
)
SELECT
  *,
  hosp_h + hosp_o + hosp_s + hosp_p + hosp_i + hosp_a + hosp_l AS hospital_score
FROM scored
"""

bq.query(hospital_sql).result()

# --- Build combined baselines table (cohort + label + LACE + HOSPITAL) ---
combined_sql = f"""
CREATE OR REPLACE TABLE `{BASELINES_TABLE}`
CLUSTER BY subject_id AS
SELECT
  cl.subject_id,
  cl.hadm_id,
  cl.label,
  l.lace_l, l.lace_a, l.lace_c, l.lace_e, l.lace_score,
  h.hosp_h, h.hosp_o, h.hosp_s, h.hosp_p, h.hosp_i, h.hosp_a, h.hosp_l,
  h.hospital_score,
  -- raw inputs for downstream debugging
  h.hgb, h.na, h.disch_service, h.has_cancer_dx, h.has_any_procedure,
  h.n_adm_365d
FROM `{LABELED_TABLE}`  cl
JOIN `{LACE_TABLE}`     l USING (subject_id, hadm_id)
JOIN `{HOSPITAL_TABLE}` h USING (subject_id, hadm_id)
"""
bq.query(combined_sql).result()

# --- HOSPITAL component diagnostics ---
diag_sql = f"""
SELECT
  COUNT(*)                                                          AS n_rows,
  MIN(hospital_score)                                               AS min_score,
  MAX(hospital_score)                                               AS max_score,
  ROUND(AVG(hospital_score), 2)                                     AS mean_score,
  ROUND(100 * AVG(IF(hosp_h > 0, 1, 0)), 1)                         AS pct_h_fires,
  ROUND(100 * AVG(IF(hosp_o > 0, 1, 0)), 1)                         AS pct_o_fires,
  ROUND(100 * AVG(IF(hosp_s > 0, 1, 0)), 1)                         AS pct_s_fires,
  ROUND(100 * AVG(IF(hosp_p > 0, 1, 0)), 1)                         AS pct_p_fires,
  ROUND(100 * AVG(IF(hosp_i > 0, 1, 0)), 1)                         AS pct_i_fires,
  ROUND(100 * AVG(IF(hosp_a > 0, 1, 0)), 1)                         AS pct_a_fires,
  ROUND(100 * AVG(IF(hosp_l > 0, 1, 0)), 1)                         AS pct_l_fires,
  ROUND(100 * AVG(IF(hgb IS NULL, 1, 0)), 1)                        AS pct_hgb_missing,
  ROUND(100 * AVG(IF(na  IS NULL, 1, 0)), 1)                        AS pct_na_missing
FROM `{HOSPITAL_TABLE}`
"""
d = bq.query(diag_sql).result().to_dataframe().iloc[0]

print(f"Built {HOSPITAL_TABLE}")
print(f"  rows:         {int(d.n_rows):,}")
print(f"  score range:  {int(d.min_score)}–{int(d.max_score)}  (theoretical 0–13)")
print(f"  mean score:   {d.mean_score}")
print(f"  component fire rates (%):")
print(f"    H={d.pct_h_fires}  O={d.pct_o_fires}  S={d.pct_s_fires}  "
      f"P={d.pct_p_fires}  I={d.pct_i_fires}  A={d.pct_a_fires}  L={d.pct_l_fires}")
print(f"  missing labs (%): Hgb={d.pct_hgb_missing}  Na={d.pct_na_missing}")
print(f"Built {BASELINES_TABLE}")

# --- Pull scores + label and compute AUROC / AUPRC ---------------------
eval_df = bq.query(
    f"SELECT label, lace_score, hospital_score FROM `{BASELINES_TABLE}`"
).result().to_dataframe()

y          = eval_df["label"].to_numpy()
lace       = eval_df["lace_score"].to_numpy()
hospital   = eval_df["hospital_score"].to_numpy()
prevalence = y.mean()

results = []
for name, scores in [("LACE", lace), ("HOSPITAL", hospital)]:
    auroc = roc_auc_score(y, scores)
    auprc = average_precision_score(y, scores)
    results.append((name, auroc, auprc))

print("\n--- Discriminative floor (n = {:,}, prevalence = {:.2%}) ---".format(len(y), prevalence))
print(f"{'Model':<10} {'AUROC':>8} {'AUPRC':>8}  (chance AUPRC = {prevalence:.3f})")
for name, auroc, auprc in results:
    print(f"{name:<10} {auroc:>8.4f} {auprc:>8.4f}")


## Part 4 — Deterministic patient-level splits

Builds `readmission.cohort_splits`: one row per distinct `subject_id`,
labeled with a `split` ∈ {`train`, `val`, `test`, `demo`}.

### Why hash, not random shuffle

A deterministic hash of `subject_id` removes all bookkeeping. Same
patient → same split, today, tomorrow, in CI, in production — without
saving a random seed or an assignment table. The function itself is
the seed.

```text
ABS(MOD(FARM_FINGERPRINT(CAST(subject_id AS STRING)), 1000))  →  bucket ∈ [0, 999]
```

`FARM_FINGERPRINT` is uniform, so buckets fill evenly (~0.1% of
patients each). Slicing the 0–999 line into regions yields the four
splits.

### Bucket layout

| Range          | Split   | Target % | Purpose                                          |
|----------------|---------|----------|--------------------------------------------------|
| `[0, 5)`       | `demo`  | ~0.5%    | Held-out patients for the Clinical Copilot UI / production simulation. Never seen by training, validation, or final test scoring. Drives drift + outcome-accuracy monitoring. |
| `[5, 705)`     | `train` | 70%      | Model fitting.                                   |
| `[705, 855)`   | `val`   | 15%      | Hyperparameter tuning, early-stopping, model selection. |
| `[855, 1000)`  | `test`  | ~14.5%   | Final, locked-in evaluation. Touched once per model version. |

### Leakage discipline

- Grouping unit = `subject_id`. All admissions for one patient land in
  exactly one split. The hash is computed on `subject_id` alone, so
  every `hadm_id` for that patient inherits the same bucket.
- Training unit = `hadm_id`. The model fits and is scored on
  admission-level rows. `subject_id` is metadata, never a feature.
- Demo split is sealed: it never enters training or final scoring.

The cell below builds `cohort_splits` and asserts that no
`subject_id` appears in more than one split. The follow-on cell
re-evaluates LACE/HOSPITAL on the `test` split alone — that number is
the locked-in discriminative floor every future model is compared
against.


In [ ]:
# Cell 7 — Build the deterministic patient-level splits table.
#
# One row per distinct subject_id.  bucket = ABS(MOD(FARM_FINGERPRINT, 1000)).
# Sliced into demo / train / val / test per the layout in Part 4.
#
# Self-contained: redefines the config + table refs from Cell 1 so this
# cell can run without re-executing upstream CRUD.
from google.cloud import bigquery

PROJECT_ID    = "enterprise-clinical-copilot"
LOCATION      = "US"
DEST          = f"{PROJECT_ID}.readmission"
COHORT_TABLE  = f"{DEST}.cohort_admissions"
LABELED_TABLE = f"{DEST}.cohort_labeled"
SPLITS_TABLE  = f"{DEST}.cohort_splits"

bq = bigquery.Client(project=PROJECT_ID, location=LOCATION)

splits_sql = f"""
CREATE OR REPLACE TABLE `{SPLITS_TABLE}`
CLUSTER BY subject_id AS
WITH hashed AS (
  SELECT DISTINCT
    subject_id,
    ABS(MOD(FARM_FINGERPRINT(CAST(subject_id AS STRING)), 1000)) AS bucket
  FROM `{COHORT_TABLE}`
)
SELECT
  subject_id,
  bucket,
  CASE
    WHEN bucket <    5 THEN 'demo'
    WHEN bucket <  705 THEN 'train'
    WHEN bucket <  855 THEN 'val'
    ELSE                    'test'
  END AS split
FROM hashed
"""

bq.query(splits_sql).result()

# --- Leakage assertion: no subject_id may span splits ------------------
leak_sql = f"""
SELECT COUNT(*) AS n_leaky_patients
FROM (
  SELECT subject_id
  FROM `{SPLITS_TABLE}`
  GROUP BY subject_id
  HAVING COUNT(DISTINCT split) > 1
)
"""
n_leaky = int(bq.query(leak_sql).result().to_dataframe().iloc[0]["n_leaky_patients"])
assert n_leaky == 0, f"LEAKAGE: {n_leaky} subject_id(s) span multiple splits"

# --- Per-split diagnostics: patients, admissions, prevalence -----------
diag_sql = f"""
SELECT
  s.split,
  COUNT(DISTINCT s.subject_id)                         AS n_patients,
  COUNT(*)                                             AS n_admissions,
  ROUND(100 * COUNTIF(cl.label = 1) / COUNT(*), 2)     AS prevalence_pct,
  ROUND(AVG(adm_per_pt.n), 2)                          AS mean_admits_per_patient
FROM `{SPLITS_TABLE}` s
JOIN `{LABELED_TABLE}` cl USING (subject_id)
JOIN (
  SELECT subject_id, COUNT(*) AS n
  FROM `{COHORT_TABLE}`
  GROUP BY subject_id
) adm_per_pt USING (subject_id)
GROUP BY s.split
ORDER BY
  CASE s.split
    WHEN 'train' THEN 1
    WHEN 'val'   THEN 2
    WHEN 'test'  THEN 3
    WHEN 'demo'  THEN 4
  END
"""
splits_df = bq.query(diag_sql).result().to_dataframe()

print(f"Built {SPLITS_TABLE}")
print(f"  leakage check: PASS (0 patients span splits)")
print()
print(splits_df.to_string(index=False))


In [ ]:
# Cell 8 — Locked-in baselines on the TEST split only.
#
# This is the discriminative floor every future model must beat.
# Numbers are computed once on test and recorded; in Phase B they are
# logged to Vertex AI as run `baseline-v1`.
#
# Self-contained: redefines the config + table refs so this cell can run
# without re-executing upstream CRUD.
import pandas as pd
from google.cloud import bigquery
from sklearn.metrics import roc_auc_score, average_precision_score

PROJECT_ID      = "enterprise-clinical-copilot"
LOCATION        = "US"
DEST            = f"{PROJECT_ID}.readmission"
BASELINES_TABLE = f"{DEST}.cohort_baselines"
SPLITS_TABLE    = f"{DEST}.cohort_splits"

bq = bigquery.Client(project=PROJECT_ID, location=LOCATION)

test_sql = f"""
SELECT b.label, b.lace_score, b.hospital_score
FROM `{BASELINES_TABLE}` b
JOIN `{SPLITS_TABLE}`     s USING (subject_id)
WHERE s.split = 'test'
"""
test_df = bq.query(test_sql).result().to_dataframe()

n = len(test_df)
prev = test_df["label"].mean()

results = []
for name, col in [("LACE", "lace_score"), ("HOSPITAL", "hospital_score")]:
    auroc = roc_auc_score(test_df["label"], test_df[col])
    auprc = average_precision_score(test_df["label"], test_df[col])
    results.append({"baseline": name, "AUROC": round(auroc, 4), "AUPRC": round(auprc, 4)})

print(f"Test-split evaluation")
print(f"  admissions = {n:,}")
print(f"  positives  = {int(test_df['label'].sum()):,}")
print(f"  prevalence = {prev:.4f}")
print()
print(pd.DataFrame(results).to_string(index=False))
print()
print("These are the locked-in baselines for `readmission-30d` Phase B.")


## Part 5 — Log `baseline-v1` to Vertex AI Experiments

Phase B closes the data-representation work by registering the
locked-in baselines as the first run in the `readmission-30d`
experiment. Every future model — logistic regression, gradient-boosted
trees, the eventual deep model — is logged to the same experiment and
judged against this run.

### Architecture: tracking lives in `src/`, not in the notebook

Two reusable modules are introduced now (Phase B exception to the
"defer `src/` to Phase F" rule, because every later phase consumes
them):

- **`src/config.py`** — single source of truth: project ID, region,
  GCS bucket, BigQuery table FQNs, MIMIC version, the eight acute
  admission types, the four-way split contract, the experiment name,
  and a `git_sha()` provenance helper.
- **`src/tracking.py`** — thin wrapper around Vertex AI Experiments:
  `init()` (idempotent), `log_run(...)` context manager,
  `log_params` / `log_metrics`. Auto-injects `git_sha` and
  `mimic_version` into every run so provenance is never optional.

Both modules are import-clean (no GCP client construction at import
time), so they're safe to use from notebooks, Vertex pipeline
container steps, and the demo app alike.

### What gets logged for `baseline-v1`

`baseline-v1` is **scalar metrics + run params only** — no GCS
artifacts, no figures, no HTML. Visualization is deferred. The
prose interpretation of these numbers lives in
[`docs/baseline_v1_interpretation.md`](docs/baseline_v1_interpretation.md).

**Parameters** (the input contract — what the model was told):

- Table FQNs: cohort / labeled / splits / baselines.
- Cohort scale: `n_admissions_total`, `n_patients_total`,
  `prevalence_overall`.
- Per-split admission counts, patient counts, and percentages for
  train / val / test / demo.
- `split_strategy` (`FARM_FINGERPRINT(subject_id) % 1000`),
  `acute_types`, `mimic_version`, `git_sha`.
- `lace_high_risk_threshold` (10), `hospital_high_risk_threshold` (7).

**Metrics — the discriminative floor every later run is measured against:**

*Test split* (sealed; one shot per run):

- `{lace,hospital}_test_auroc`, `{lace,hospital}_test_auprc`
- `{lace,hospital}_test_{sensitivity,specificity,ppv,npv,flag_rate}`
  at literature high-risk cutoffs (LACE ≥ 10; HOSPITAL ≥ 7) — these
  are the deployed-rule screening characteristics any new model must
  beat at the same flag-rate to claim clinical lift.
- `prevalence_test`, `chance_auprc_test`, `n_test`.

*Val split* (used during model selection):

- `{lace,hospital}_val_auroc`, `{lace,hospital}_val_auprc`
- `prevalence_val`, `chance_auprc_val`, `n_val`.

### Conventions for downstream runs

- One experiment, many runs. Run names follow `<family>-v<n>`
  (`logreg-v1`, `gbt-v1`, `gbt-v2`, …).
- Parameters describe the **input contract** (cohort, splits, feature
  set version). Metrics describe **performance**. Never mix them.
- Test metrics are computed once per run on the sealed `test` split.
  Val metrics are logged alongside under `*_val_*` keys.
- `chance_auprc_{test,val}` is logged on every run so AUPRC stays
  interpretable across cohort/prevalence drift.


In [ ]:
# Cell 9 — Log baseline-v1 to Vertex AI Experiments.
#
# Establishes the discriminative floor that every later model run
# (logreg-v1, gbt-v1, ...) is judged against. Logs scalar metrics only
# — no GCS artifacts. Visualization and reporting are deferred.
#
# Metric surface (locked in here as the comparison contract):
#
#   On test split (sealed; one shot per run):
#     {lace,hospital}_test_auroc
#     {lace,hospital}_test_auprc
#     {lace,hospital}_test_{sensitivity,specificity,ppv,npv,flag_rate}
#         at literature high-risk cutoffs (LACE >= 10; HOSPITAL >= 7).
#     prevalence_test, chance_auprc_test, n_test
#
#   On val split (used during model selection):
#     {lace,hospital}_val_auroc
#     {lace,hospital}_val_auprc
#     prevalence_val, chance_auprc_val, n_val
#
# References for the clinical thresholds:
#   LACE     >= 10  -> high risk : van Walraven et al., CMAJ 2010.
#   HOSPITAL >= 7   -> high risk : Donzé et al., JAMA Intern Med 2013.

import os, sys
_HERE = os.path.dirname(os.path.abspath("__file__"))
if _HERE not in sys.path:
    sys.path.insert(0, _HERE)

import pandas as pd
from google.cloud import bigquery
from sklearn.metrics import (
    roc_auc_score, average_precision_score, confusion_matrix,
)

from src import config, tracking

RUN_NAME = "baseline-v1"

# Literature high-risk cutoffs (see citations above).
THRESHOLDS = {"lace": 10, "hospital": 7}

bq = bigquery.Client(project=config.PROJECT_ID, location=config.BQ_LOCATION)

# --- 1. Cohort + split contract (run params) ---------------------------
contract_sql = f"""
WITH overall AS (
  SELECT
    COUNT(*)                   AS n_admissions_total,
    COUNT(DISTINCT subject_id) AS n_patients_total,
    AVG(label)                 AS prevalence_overall
  FROM `{config.LABELED_TABLE}`
),
per_split AS (
  SELECT
    s.split,
    COUNT(*)                     AS n_admissions,
    COUNT(DISTINCT s.subject_id) AS n_patients,
    AVG(cl.label)                AS prevalence
  FROM `{config.SPLITS_TABLE}` s
  JOIN `{config.LABELED_TABLE}` cl USING (subject_id)
  GROUP BY s.split
)
SELECT
  (SELECT n_admissions_total FROM overall) AS n_admissions_total,
  (SELECT n_patients_total   FROM overall) AS n_patients_total,
  (SELECT prevalence_overall FROM overall) AS prevalence_overall,
  MAX(IF(split='train', n_admissions, NULL)) AS n_admissions_train,
  MAX(IF(split='val',   n_admissions, NULL)) AS n_admissions_val,
  MAX(IF(split='test',  n_admissions, NULL)) AS n_admissions_test,
  MAX(IF(split='demo',  n_admissions, NULL)) AS n_admissions_demo,
  MAX(IF(split='train', n_patients,   NULL)) AS n_patients_train,
  MAX(IF(split='val',   n_patients,   NULL)) AS n_patients_val,
  MAX(IF(split='test',  n_patients,   NULL)) AS n_patients_test,
  MAX(IF(split='demo',  n_patients,   NULL)) AS n_patients_demo
FROM per_split
"""
c = bq.query(contract_sql).result().to_dataframe().iloc[0]
n_total = int(c.n_admissions_total)

params = {
    "cohort_table":       config.COHORT_TABLE,
    "labeled_table":      config.LABELED_TABLE,
    "splits_table":       config.SPLITS_TABLE,
    "baselines_table":    config.BASELINES_TABLE,
    "split_strategy":     config.SPLIT_CONTRACT.strategy,
    "acute_types":        config.ACUTE_ADMISSION_TYPES,
    "n_admissions_total": n_total,
    "n_patients_total":   int(c.n_patients_total),
    "prevalence_overall": round(float(c.prevalence_overall), 4),
    "n_admissions_train": int(c.n_admissions_train),
    "n_admissions_val":   int(c.n_admissions_val),
    "n_admissions_test":  int(c.n_admissions_test),
    "n_admissions_demo":  int(c.n_admissions_demo),
    "n_patients_train":   int(c.n_patients_train),
    "n_patients_val":     int(c.n_patients_val),
    "n_patients_test":    int(c.n_patients_test),
    "n_patients_demo":    int(c.n_patients_demo),
    "split_train_pct":    round(100 * c.n_admissions_train / n_total, 2),
    "split_val_pct":      round(100 * c.n_admissions_val   / n_total, 2),
    "split_test_pct":     round(100 * c.n_admissions_test  / n_total, 2),
    "split_demo_pct":     round(100 * c.n_admissions_demo  / n_total, 2),
    "lace_high_risk_threshold":     THRESHOLDS["lace"],
    "hospital_high_risk_threshold": THRESHOLDS["hospital"],
}

# --- 2. Pull scored rows for test and val splits ----------------------
def _scores(split: str) -> pd.DataFrame:
    sql = f"""
    SELECT b.label, b.lace_score, b.hospital_score
    FROM `{config.BASELINES_TABLE}` b
    JOIN `{config.SPLITS_TABLE}`     s USING (subject_id)
    WHERE s.split = @split
    """
    job = bq.query(
        sql,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ScalarQueryParameter("split", "STRING", split),
            ],
        ),
    )
    return job.result().to_dataframe()

test_df = _scores("test")
val_df  = _scores("val")

# --- 3. Compute the evaluation-floor metrics --------------------------
def _ranking_metrics(df: pd.DataFrame, split_tag: str) -> dict[str, float]:
    """AUROC / AUPRC for both scorers on a single split."""
    y = df["label"].astype(int).values
    out = {
        f"prevalence_{split_tag}":   float(y.mean()),
        f"chance_auprc_{split_tag}": float(y.mean()),
        f"n_{split_tag}":            float(len(df)),
    }
    for scorer, col in (("lace", "lace_score"), ("hospital", "hospital_score")):
        s = df[col].astype(float).values
        out[f"{scorer}_{split_tag}_auroc"] = float(roc_auc_score(y, s))
        out[f"{scorer}_{split_tag}_auprc"] = float(average_precision_score(y, s))
    return out

def _clinical_cutoff_metrics(df: pd.DataFrame) -> dict[str, float]:
    """Sens/spec/PPV/NPV/flag-rate at the literature high-risk cutoffs.

    Computed on test only — these are the deployed-rule screening
    characteristics every future model will be benchmarked against.
    """
    y = df["label"].astype(int).values
    n = len(y)
    out: dict[str, float] = {}
    for scorer, col in (("lace", "lace_score"), ("hospital", "hospital_score")):
        thr = THRESHOLDS[scorer]
        pred = (df[col].astype(float).values >= thr).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
        out[f"{scorer}_test_sensitivity"] = tp / (tp + fn) if (tp + fn) else 0.0
        out[f"{scorer}_test_specificity"] = tn / (tn + fp) if (tn + fp) else 0.0
        out[f"{scorer}_test_ppv"]         = tp / (tp + fp) if (tp + fp) else 0.0
        out[f"{scorer}_test_npv"]         = tn / (tn + fn) if (tn + fn) else 0.0
        out[f"{scorer}_test_flag_rate"]   = (tp + fp) / n if n else 0.0
    return out

metrics: dict[str, float] = {}
metrics.update(_ranking_metrics(test_df, "test"))
metrics.update(_ranking_metrics(val_df,  "val"))
metrics.update(_clinical_cutoff_metrics(test_df))

# --- 4. Open the run, log params + metrics (no artifacts) -------------
with tracking.log_run(RUN_NAME, params=params):
    tracking.log_metrics(metrics)

# --- 5. Echo summary --------------------------------------------------
print(f"Logged run `{RUN_NAME}` under experiment `{config.EXPERIMENT_NAME}`")
print(f"  region: {config.VERTEX_REGION}")
print()
print("Test-split ranking metrics (the discriminative floor):")
for scorer in ("lace", "hospital"):
    print(f"  {scorer:8s}  AUROC={metrics[f'{scorer}_test_auroc']:.4f}"
          f"   AUPRC={metrics[f'{scorer}_test_auprc']:.4f}")
print(f"  chance    AUPRC={metrics['chance_auprc_test']:.4f}"
      f"  (prevalence_test, n_test={int(metrics['n_test']):,})")
print()
print("Val-split ranking metrics (for model-selection comparisons):")
for scorer in ("lace", "hospital"):
    print(f"  {scorer:8s}  AUROC={metrics[f'{scorer}_val_auroc']:.4f}"
          f"   AUPRC={metrics[f'{scorer}_val_auprc']:.4f}")
print(f"  chance    AUPRC={metrics['chance_auprc_val']:.4f}"
      f"  (prevalence_val,  n_val ={int(metrics['n_val']):,})")
print()
print("Test-split clinical-cutoff screening metrics:")
for scorer in ("lace", "hospital"):
    thr = THRESHOLDS[scorer]
    print(f"  {scorer:8s}  rule: score >= {thr}"
          f"   sens={metrics[f'{scorer}_test_sensitivity']:.3f}"
          f"   spec={metrics[f'{scorer}_test_specificity']:.3f}"
          f"   PPV={metrics[f'{scorer}_test_ppv']:.3f}"
          f"   NPV={metrics[f'{scorer}_test_npv']:.3f}"
          f"   flag-rate={metrics[f'{scorer}_test_flag_rate']:.3f}")
print()
print(f"Console: {tracking.run_console_url(RUN_NAME)}")


Associating projects/639646300983/locations/us-east1/metadataStores/default/contexts/readmission-30d-baseline-v1 to Experiment: readmission-30d


Logged run `baseline-v1` under experiment `readmission-30d`
  region:    us-east1
  artifact:  gs://enterprise-clinical-copilot-mlops/artifacts/baseline-v1/test_scores.csv

Metrics:
  lace_auroc         0.6466
  lace_auprc         0.2892
  hospital_auroc     0.6829
  hospital_auprc     0.3308
  chance_auprc       0.1913
  prevalence_test    0.1913
  n_test             59792.0

Console: https://console.cloud.google.com/vertex-ai/experiments/locations/us-east1/experiments/readmission-30d/runs?project=enterprise-clinical-copilot
